In [5]:
import tensorflow as tf

print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

GPU tersedia: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
!nvidia-smi

Wed Sep  9 04:20:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Tahap 2 — Download Poster Tambahan (MovieGenre.csv)

Notebook ini mengunduh poster film tambahan dari URL yang sudah tersedia di `MovieGenre.csv`
(kolom `Poster`), memprioritaskan genre minoritas (Animation, Action, Horror, Adventure)
supaya dataset akhir tidak setimpang sample awal (997 gambar, Drama 601 vs Animation 28).

**Sebelum run:** pastikan `MovieGenre.csv` sudah ada di direktori kerja Colab
(via upload manual, atau mount Google Drive).


**Catatan path:** notebook ini diasumsikan dijalankan dari folder `notebooks/` pada struktur project lokal (bukan root Colab). `MovieGenre.csv` dibaca dari `../data/raw/`, poster mentah disimpan ke `../data/raw/`, dan log unduhan ke `../outputs/logs/`. Kalau dijalankan langsung di Colab (bukan lewat mount project ini), sesuaikan ulang variabel `CSV_PATH` dan `OUT_DIR` ke path yang sesuai di sana.

In [ ]:
import os
import time
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

## Konfigurasi

In [ ]:
import sys

# Deteksi otomatis: Colab atau lokal
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files

    # Upload MovieGenre.csv kalau belum ada
    if not os.path.exists("MovieGenre.csv"):
        print("Upload file MovieGenre.csv dari komputer lokal:")
        uploaded = files.upload()

    CSV_PATH = "MovieGenre.csv"
    OUT_DIR = "raw"
else:
    CSV_PATH = "../data/raw/MovieGenre.csv"
    OUT_DIR = "../data/raw"

TARGET_GENRES = ["Action", "Adventure", "Animation", "Comedy",
                  "Drama", "Horror", "Romance"]
MAX_PER_GENRE = 400                  # atur sesuai kebutuhan/waktu unduh
TIMEOUT = 8
MAX_WORKERS = 16

os.makedirs(OUT_DIR, exist_ok=True)
print(f"CSV_PATH = {CSV_PATH}")
print(f"OUT_DIR  = {OUT_DIR}")

## Load &amp; Filter Dataset

Ambil baris dengan genre &amp; link poster valid, lalu simpan hanya genre yang termasuk 7 target.

In [ ]:
df = pd.read_csv(CSV_PATH, encoding="latin1")
df = df.dropna(subset=["Genre", "Poster"])
df["genre_list"] = df["Genre"].astype(str).str.split("|")
df["target_genres"] = df["genre_list"].apply(
    lambda gl: [g for g in gl if g in TARGET_GENRES]
)
df = df[df["target_genres"].apply(len) > 0].copy()

print("Total baris dengan minimal 1 genre target:", len(df))

## Prioritaskan Genre Minoritas

Kalau diunduh urut apa adanya, kuota `MAX_PER_GENRE` akan cepat habis oleh Drama/Comedy
(paling banyak jumlahnya), sementara Animation/Action/Horror/Adventure (paling sedikit)
tidak kebagian jatah. Jadi baris dengan genre langka diproses lebih dulu.

In [ ]:
genre_counts = {g: 0 for g in TARGET_GENRES}
minority_order = ["Animation", "Action", "Horror", "Adventure",
                   "Romance", "Comedy", "Drama"]

def genre_priority(genres):
    ranks = [minority_order.index(g) for g in genres if g in minority_order]
    return min(ranks) if ranks else 99

df["priority"] = df["target_genres"].apply(genre_priority)
df = df.sort_values("priority")

selected_rows = []
for _, row in df.iterrows():
    gl = row["target_genres"]
    if any(genre_counts[g] < MAX_PER_GENRE for g in gl):
        selected_rows.append(row)
        for g in gl:
            genre_counts[g] += 1

print("Rencana unduh:", len(selected_rows), "poster")
print("Estimasi per genre (bisa overlap krn multi-label):", genre_counts)

## Fungsi Download Satu Poster

In [ ]:
def download_one(row):
    imdb_id = str(row["imdbId"])
    url = row["Poster"]
    out_path = os.path.join(OUT_DIR, f"{imdb_id}.jpg")
    if os.path.exists(out_path):
        return imdb_id, "skip_exists"
    try:
        r = requests.get(url, timeout=TIMEOUT)
        if r.status_code == 200 and r.content:
            with open(out_path, "wb") as f:
                f.write(r.content)
            return imdb_id, "ok"
        else:
            return imdb_id, f"http_{r.status_code}"
    except Exception as e:
        err_name = type(e).__name__
        return imdb_id, "error_" + err_name

## Jalankan Download Paralel

Memakai 16 thread sekaligus supaya tidak menunggu satu-satu.

In [ ]:
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(download_one, row): row for _, row in
               pd.DataFrame(selected_rows).iterrows()}
    for i, fut in enumerate(as_completed(futures), 1):
        imdb_id, status = fut.result()
        results.append((imdb_id, status))
        if i % 200 == 0:
            print(f"progress: {i}/{len(selected_rows)}")

## Ringkasan &amp; Log Hasil

Catat status tiap unduhan (berhasil / link mati / error) supaya bisa diaudit.

In [ ]:
res_df = pd.DataFrame(results, columns=["imdbId", "status"])
print(res_df["status"].value_counts())

LOG_DIR = "logs" if IN_COLAB else "../outputs/logs"
os.makedirs(LOG_DIR, exist_ok=True)
log_path = os.path.join(LOG_DIR, "download_log_1b.csv")
res_df.to_csv(log_path, index=False)

ok_ids = set(res_df.loc[res_df["status"] == "ok", "imdbId"])
print("Berhasil diunduh:", len(ok_ids), "poster baru")
print(f"File log tersimpan di {log_path} untuk audit link mati.")

## Verifikasi Distribusi Genre Riil

Target `MAX_PER_GENRE` dihitung dari ketersediaan baris di CSV **sebelum** download,
bukan dari hasil download yang sukses. Karena ada link mati (`http_404`, dll), jumlah
riil per genre setelah download bisa lebih kecil dari target — terutama untuk genre
minoritas (Animation, Action). Cell ini juga mengecek apakah ada `imdbId` duplikat
di dataset final.

In [ ]:
import os
from collections import Counter

downloaded_ids = set(f.split(".")[0] for f in os.listdir(OUT_DIR))
final_df = df[df["imdbId"].astype(str).isin(downloaded_ids)]

# cek duplikat imdbId
dup_count = final_df["imdbId"].duplicated().sum()
print("Baris dengan imdbId duplikat:", dup_count)

# hitung distribusi genre riil (setelah drop duplikat)
final_df = final_df.drop_duplicates(subset="imdbId")
c = Counter()
for gl in final_df["target_genres"]:
    for g in gl:
        c[g] += 1
print("Distribusi genre riil setelah download:")
for g in TARGET_GENRES:
    print(f"  {g}: {c.get(g, 0)}")

# Tahap 3 — Gabungkan Semua Poster & Bersihkan Duplikat (Deduplikasi 3 Lapis)

Tahap ini memvalidasi integritas file poster, mendeteksi duplikasi melalui 3 lapis:
1. **Lapis 1:** `imdbId` yang sama pada dataset.
2. **Lapis 2:** Byte Hash (MD5) untuk mendeteksi file poster identik secara biner.
3. **Lapis 3:** Visual Perceptual Hash (dHash) untuk mendeteksi poster yang secara visual sama persis.

File yang duplikat atau corrupt dipindahkan ke folder `quarantine/` (non-destruktif), dan data bersih disimpan ke `data/processed/clean_posters_metadata.csv`.

In [ ]:
import os
import sys
import pandas as pd

# Konfigurasi Path Tahap 3
LOG_DIR = "logs" if IN_COLAB else "../outputs/logs"
os.makedirs(LOG_DIR, exist_ok=True)
QUARANTINE_DIR = "quarantine" if IN_COLAB else "../data/quarantine"
PROCESSED_DIR = "processed" if IN_COLAB else "../data/processed"
CLEAN_META_PATH = os.path.join(PROCESSED_DIR, "clean_posters_metadata.csv")
AUDIT_LOG_PATH = os.path.join(LOG_DIR, "stage3_dedup_audit.csv")

os.makedirs(QUARANTINE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"OUT_DIR         : {OUT_DIR}")
print(f"QUARANTINE_DIR  : {QUARANTINE_DIR}")
print(f"CLEAN_META_PATH : {CLEAN_META_PATH}")
print(f"AUDIT_LOG_PATH  : {AUDIT_LOG_PATH}")

In [ ]:
# Import fungsi deduplikasi dari modul reusable src/utils/dedup.py
for _p in ("../src", "src"):
    if os.path.exists(os.path.join(_p, "utils", "dedup.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
from utils.dedup import run_stage3_pipeline
print("Modul dedup dimuat dari:", os.path.abspath(_p))

In [ ]:
# Eksekusi Deduplikasi 3 Lapis & Pembersihan (via run_stage3_pipeline)
df_valid, df_audit = run_stage3_pipeline(
    posters_dir=OUT_DIR,
    df_metadata=df,
    quarantine_dir=QUARANTINE_DIR,
    output_metadata_path=CLEAN_META_PATH,
    log_audit_path=AUDIT_LOG_PATH,
)

# Tahap 4 — Cek Jumlah Akhir per Genre

Validasi distribusi genre final **setelah gabungan poster lama + baru melewati deduplikasi 3 lapis (Tahap 3)**.
Berbeda dengan verifikasi di akhir Tahap 2 (cek hasil download), tahap ini memakai
`clean_posters_metadata.csv` — dataset bersih yang akan jadi input Tahap 5+.

Keputusan setelah tahap ini:
- Distribusi cukup seimbang → lanjut Tahap 5 (bersihkan data).
- Masih timpang ekstrem → unduh poster tambahan (kembali ke Tahap 2) atau terima ketimpangan
  dan catat sebagai limitasi di skripsi.

In [ ]:
# Tahap 4: Distribusi genre final dari metadata bersih hasil Tahap 3
from collections import Counter

df_clean = pd.read_csv(CLEAN_META_PATH)

print(f"Total poster bersih     : {len(df_clean)}")
print(f"Baris duplikat imdbId   : {df_clean['imdbId'].duplicated().sum()}")
print()

# Distribusi genre (kolom Genre dari metadata gabungan)
c = Counter()
for genres in df_clean["Genre"].dropna():
    for g in str(genres).split(","):
        g = g.strip()
        if g in TARGET_GENRES:
            c[g] += 1

print("Distribusi genre final (7 genre target):")
for g in TARGET_GENRES:
    n = c.get(g, 0)
    bar = "#" * max(1, n // 25)
    print(f"  {g:<12} {n:>5}  {bar}")

total = sum(c.values())
print(f"\nTotal label genre (multi-label, satu poster bisa punya >1): {total}")
mx, mn = max(c.values()), min(c.values())
print(f"Rasio genre terbesar/terkecil: {mx/mn:.1f}:1  (max={mx}, min={mn})")
if mx/mn > 3:
    print("WARNING: distribusi masih timpang (>3:1). Pertimbangkan tambah data genre minoritas atau catat sebagai limitasi.")
else:
    print("Distribusi cukup seimbang (<3:1). Lanjut Tahap 5.")

# Tahap 5 — Split Data & Input Pipeline (Persiapan Training)

Dataset bersih hasil Tahap 3 (1535 poster, 7 genre, multi-label) dipecah menjadi
train/val/test 70/15/15 secara terstratifikasi (genre paling langka per baris),
lalu disiapkan pipeline `tf.data` dan arsitektur model:

- **Model utama:** MobileNetV2 transfer learning (feature extractor ImageNet + head sigmoid).
- **Baseline pembanding:** CNN kecil from-scratch (`build_baseline_cnn`), opsional untuk bab hasil.

Kode reusable ada di `src/utils/split.py`, `src/dataloader.py`, `src/models.py`.
Loss: `binary_crossentropy` + sigmoid (multi-label, bukan softmax).

In [8]:
# Tahap 5: Konfigurasi path + split terstratifikasi
import os
import shutil
import sys

import pandas as pd

IN_COLAB = "google.colab" in sys.modules

POSTERS_DIR = "posters" if IN_COLAB else "../data/raw"
PROCESSED_DIR = "processed" if IN_COLAB else "../data/processed"
MODELS_DIR = "models" if IN_COLAB else "../models"
SPLITS_PATH = os.path.join(PROCESSED_DIR, "splits.csv")
os.makedirs(MODELS_DIR, exist_ok=True)

# Fallback Colab: upload src/ kalau belum ada di runtime
if IN_COLAB and not os.path.exists("src/utils/split.py"):
    from google.colab import files as gfiles
    print("Upload 3 file dari src/ lokal: split.py, dataloader.py, models.py")
    uploaded = gfiles.upload()
    os.makedirs("src/utils", exist_ok=True)
    for name in uploaded:
        dest = os.path.join("src/utils", name) if name == "split.py" else os.path.join("src", name)
        shutil.move(name, dest)
    print("File src/ ditempatkan.")
elif not os.path.exists(os.path.join("../src", "utils", "split.py")) and not os.path.exists(os.path.join("src", "utils", "split.py")):
    raise FileNotFoundError("Folder src/ tidak ditemukan — jalankan dari root project atau notebooks/")

for _p in ("../src", "src"):
    if os.path.exists(os.path.join(_p, "utils", "split.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
from utils.split import GENRES, make_splits, verify_splits

df_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "clean_posters_metadata.csv"))

splits = make_splits(df_clean, seed=42)
splits.to_csv(SPLITS_PATH, index=False)

print(splits["split"].value_counts())
print(verify_splits(splits, df_clean))
print(f"Split tersimpan di: {SPLITS_PATH}")

Upload 3 file dari src/ lokal: split.py, dataloader.py, models.py


KeyboardInterrupt: 

In [7]:
# Tahap 5: Build tf.data datasets + sanity check
import tensorflow as tf

from utils.split import parse_genres
from dataloader import build_dataset, multihot_labels

df_split = df_clean.merge(splits[["imdbId", "split"]], on="imdbId")
df_split["genres"] = df_split["target_genres"].apply(parse_genres)

def make_subset(name):
    part = df_split[df_split["split"] == name]
    return build_dataset(
        POSTERS_DIR,
        part["filename"].tolist(),
        multihot_labels(part["genres"], GENRES),
        training=(name == "train"),
    )

train_ds, val_ds, test_ds = make_subset("train"), make_subset("val"), make_subset("test")

for images, labels in train_ds.take(1):
    print("Batch shape:", images.shape, labels.shape)
    print("Label range:", float(tf.reduce_min(labels)), "-", float(tf.reduce_max(labels)))
print("GPU:", tf.config.list_physical_devices("GPU"))

ModuleNotFoundError: No module named 'utils'

In [ ]:
# Tahap 5: Train model utama (MobileNetV2 transfer learning)
from models import build_transfer_model

model = build_transfer_model(num_classes=len(GENRES))
model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    ],
)

model.save(os.path.join(MODELS_DIR, "mobilenetv2_stage5.keras"))
print("Model tersimpan.")